# 第 8 课：Supervisor 多 Agent 协作

预计用时：90–120 分钟  
适合人群：完成上一课的零基础学习者；本 Notebook 也包含独立运行所需的准备代码。

## 学习目标

- 为多个角色设计清晰交接契约
- 用 Supervisor 决定返工路径
- 限制返工轮数并处理结构化输出失败

## 学习方式

按顺序运行每个代码单元格。先阅读预测结果，再运行验证；遇到报错先看本课“常见问题”，不要直接跳过。带有真实模型或外网请求的示例默认注释，确认 API Key 与费用后再启用。


## 先别急着看代码

这一课只做三件事：

1. 让研究员找资料
2. 让作者只根据资料写作
3. 让审稿人决定通过还是返工

第一次学习时，只要求能按顺序运行并用自己的话解释结果。类、类型注解和异常处理的全部细节，不需要一次记住。


## 本课术语卡

- **多 Agent**：多个职责不同的步骤协作
- **Supervisor**：决定下一步交给谁的调度者
- **交接契约**：角色之间约定好的输入和输出
- **返工上限**：防止反复修改没有尽头

看到陌生英文时先回到这里。一个术语只需要先记住一句话。


## 推荐学习动作

每个代码格都按这个顺序学习：

1. 先读上方说明，只找“输入”和“输出”。
2. 不修改代码，按 `Shift + Enter` 运行。
3. 看实际结果是否符合说明。
4. 只改一个最小值，再运行一次。

如果报 `NameError`，通常是漏跑了前面的格子；选择 **Restart Kernel and Run All** 可以从头重来。


## 0. 最小热身：先用三个普通函数模拟三个角色

这一段与后面完整工程代码相互独立。先运行它，立刻看到结果。


In [ ]:
def researcher(topic):
    return [f"关于{topic}的资料 1"]

def writer(topic, sources):
    return f"根据 {sources[0]} 写成的初稿"

def reviewer(draft):
    return {"pass": len(draft) > 10, "issues": []}

sources = researcher("Agent")
draft = writer("Agent", sources)
review = reviewer(draft)
print(sources)
print(draft)
print(review)


**你应该观察到什么？**

角色并不神秘：它们只是输入输出明确的步骤。之后再把普通函数换成模型调用。

如果结果符合说明，再继续下面的完整版本。


## 1. 先理解概念

多 Agent 的价值来自专业分工和可检查的交接，而不是角色数量。Researcher 提供来源，Writer 只能基于来源写作，Reviewer 给出结构化意见，Supervisor 决定完成、重写或重新研究。

### 本课路线

1. 定义共享写作状态
2. 实现研究、写作和审稿节点
3. 解析并兜底审稿 JSON
4. 实现 Supervisor 条件路由
5. 编译图并检查最终状态


## 2. 运行前检查

1. 从项目根目录启动 Jupyter Lab。
2. 选择项目 `.venv` 对应的 Python 内核。
3. 若本课调用百炼，先在启动 Jupyter 的终端设置 `DASHSCOPE_API_KEY`。
4. 不要把 Key 粘贴到单元格、截图或 Git 提交中。

> 下方“准备代码”可能与前课重复，这是为了保证每个 Notebook 都能单独运行。初学时建议展开阅读，熟悉后可折叠。


### 本课额外依赖


### 现在做什么？

这一小格代码

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph


### 准备代码


### 现在做什么？

这一小格代码

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
# %pip install -q openai pydantic>=2.7 httpx fastapi uvicorn fastmcp langgraph langfuse ragas numpy pytest

import os
from dotenv import load_dotenv

load_dotenv()

# 推荐在启动 Jupyter 前设置：
# Windows PowerShell: $env:DASHSCOPE_API_KEY='sk-...'
# macOS/Linux:       export DASHSCOPE_API_KEY='sk-...'

BAILIAN_API_KEY = os.getenv('DASHSCOPE_API_KEY', '')
BAILIAN_BASE_URL = os.getenv(
    'BAILIAN_BASE_URL',
    'https://dashscope.aliyuncs.com/compatible-mode/v1',
)
BAILIAN_MODEL = os.getenv('BAILIAN_MODEL', 'qwen-plus')
BAILIAN_EMBEDDING_MODEL = os.getenv('BAILIAN_EMBEDDING_MODEL', 'text-embedding-v4')

print('模型:', BAILIAN_MODEL)
print('Base URL:', BAILIAN_BASE_URL)
print('API Key:', '已配置' if BAILIAN_API_KEY else '未配置（调用模型前必须设置）')


### 准备代码


### 现在做什么？

这一小格代码

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
from __future__ import annotations

import asyncio
import json
import logging
import math
import sqlite3
import time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Awaitable, Callable, Literal, TypedDict

import httpx
import numpy as np
from openai import AsyncOpenAI
from pydantic import BaseModel, ConfigDict, Field, ValidationError

WORKSPACE = (Path.cwd() / 'agent_workspace').resolve()
WORKSPACE.mkdir(exist_ok=True)

def require_api_key() -> None:
    if not BAILIAN_API_KEY:
        raise RuntimeError('请先设置环境变量 DASHSCOPE_API_KEY，然后重新运行配置单元格。')

client = AsyncOpenAI(api_key=BAILIAN_API_KEY or 'missing', base_url=BAILIAN_BASE_URL)
print('工作目录:', WORKSPACE)


### 准备代码


### 现在做什么？

先给 Agent 装上四个保险丝：最多走几步、模型最多等多久、工具最多等多久、整个任务最多等多久。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
class AgentLimits(BaseModel):
    model_config = ConfigDict(extra='forbid')
    max_steps: int = Field(default=8, ge=1, le=30)
    model_timeout_s: float = Field(default=45, gt=0, le=300)
    tool_timeout_s: float = Field(default=15, gt=0, le=120)
    total_timeout_s: float = Field(default=120, gt=0, le=600)


### 现在做什么？

无论工具成功还是失败，都使用同一种返回格式，后面的代码就不必猜测结果长什么样。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
class ToolResult(BaseModel):
    ok: bool
    data: Any = None
    error: str | None = None
    retryable: bool = False


### 现在做什么？

先准备变量和依赖

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
ToolHandler = Callable[[BaseModel], Awaitable[Any]]


### 现在做什么？

这像一张工具登记卡：名字、用途、参数格式和真正执行的函数都写在一起。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
@dataclass
class RegisteredTool:
    name: str
    description: str
    args_model: type[BaseModel]
    handler: ToolHandler
    side_effect: bool = False

    def openai_schema(self) -> dict[str, Any]:
        schema = self.args_model.model_json_schema()
        schema['additionalProperties'] = False
        return {
            'type': 'function',
            'function': {
                'name': self.name,
                'description': self.description,
                'parameters': schema,
            },
        }


### 现在做什么？

工具注册表像工具箱目录，负责保存工具、生成说明书并安全执行。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, RegisteredTool] = {}

    def register(self, tool: RegisteredTool) -> None:
        if tool.name in self._tools:
            raise ValueError(f'工具重复注册: {tool.name}')
        self._tools[tool.name] = tool

    @property
    def schemas(self) -> list[dict[str, Any]]:
        return [tool.openai_schema() for tool in self._tools.values()]

    async def execute(self, name: str, raw_arguments: str, timeout_s: float) -> ToolResult:
        tool = self._tools.get(name)
        if tool is None:
            return ToolResult(ok=False, error=f'未知工具: {name}', retryable=False)
        try:
            arguments = json.loads(raw_arguments or '{}')
            validated = tool.args_model.model_validate(arguments)
        except json.JSONDecodeError as exc:
            return ToolResult(ok=False, error=f'工具参数不是合法 JSON: {exc}')
        except ValidationError as exc:
            return ToolResult(ok=False, error=f'工具参数校验失败: {exc}')
        try:
            async with asyncio.timeout(timeout_s):
                value = await tool.handler(validated)
            return ToolResult(ok=True, data=value)
        except TimeoutError:
            return ToolResult(ok=False, error=f'工具 {name} 执行超时', retryable=True)
        except httpx.HTTPStatusError as exc:
            retryable = exc.response.status_code in {408, 429, 500, 502, 503, 504}
            return ToolResult(ok=False, error=f'上游 HTTP {exc.response.status_code}', retryable=retryable)
        except Exception as exc:
            return ToolResult(ok=False, error=f'{type(exc).__name__}: {exc}', retryable=False)


### 准备代码


### 现在做什么？

先规定天气工具只接收一个非空城市名。模型给出的参数也必须通过这个检查。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
class WeatherArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')
    city: str = Field(min_length=1, max_length=80)


### 现在做什么？

定义 `ExchangeArgs`：先看输入，再看返回值，函数内部细节可以第二遍再读。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
class ExchangeArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')
    amount: float = Field(gt=0, le=10_000_000)
    from_currency: str = Field(pattern=r'^[A-Za-z]{3}$')
    to_currency: str = Field(pattern=r'^[A-Za-z]{3}$')


### 现在做什么？

定义 `TodoArgs`：先看输入，再看返回值，函数内部细节可以第二遍再读。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
class TodoArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')
    action: Literal['add', 'list', 'complete', 'delete']
    title: str | None = Field(default=None, max_length=200)
    todo_id: int | None = Field(default=None, ge=1)


### 现在做什么？

定义 `LogArgs`：先看输入，再看返回值，函数内部细节可以第二遍再读。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
class LogArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')
    level: Literal['INFO', 'WARNING', 'ERROR'] = 'INFO'
    message: str = Field(min_length=1, max_length=1000)
    metadata: dict[str, Any] = Field(default_factory=dict)


### 现在做什么？

定义 `SearchArgs`：先看输入，再看返回值，函数内部细节可以第二遍再读。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
class SearchArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')
    query: str = Field(min_length=2, max_length=200)
    max_results: int = Field(default=5, ge=1, le=10)


### 现在做什么？

这个函数先把城市名换成经纬度，再查询当前天气。两次网络请求都可能失败，所以必须设置超时。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
async def get_weather(args: WeatherArgs) -> dict[str, Any]:
    async with httpx.AsyncClient(timeout=10) as http:
        geo = await http.get('https://geocoding-api.open-meteo.com/v1/search', params={
            'name': args.city, 'count': 1, 'language': 'zh', 'format': 'json'
        })
        geo.raise_for_status()
        results = geo.json().get('results') or []
        if not results:
            return {'found': False, 'city': args.city}
        place = results[0]
        weather = await http.get('https://api.open-meteo.com/v1/forecast', params={
            'latitude': place['latitude'],
            'longitude': place['longitude'],
            'current': 'temperature_2m,apparent_temperature,precipitation,weather_code',
            'timezone': 'auto',
        })
        weather.raise_for_status()
        return {'found': True, 'city': place['name'], 'country': place.get('country'), **weather.json()['current']}


### 现在做什么？

货币换算工具把币种统一成大写，并处理同币种换算这个最简单的情况。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
async def convert_currency(args: ExchangeArgs) -> dict[str, Any]:
    source, target = args.from_currency.upper(), args.to_currency.upper()
    if source == target:
        return {'amount': args.amount, 'from': source, 'to': target, 'converted': args.amount, 'rate': 1}
    async with httpx.AsyncClient(timeout=10) as http:
        response = await http.get('https://api.frankfurter.app/latest', params={'amount': args.amount, 'from': source, 'to': target})
        response.raise_for_status()
        data = response.json()
        converted = data['rates'][target]
        return {'amount': args.amount, 'from': source, 'to': target, 'converted': converted, 'rate': converted / args.amount}


### 现在做什么？

先准备变量和依赖

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
TODO_DB = WORKSPACE / 'todos.sqlite3'


### 现在做什么？

定义 `init_todo_db`：先看输入，再看返回值，函数内部细节可以第二遍再读。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
def init_todo_db() -> None:
    with sqlite3.connect(TODO_DB) as conn:
        conn.execute('CREATE TABLE IF NOT EXISTS todos (id INTEGER PRIMARY KEY, title TEXT NOT NULL, done INTEGER NOT NULL DEFAULT 0)')


### 现在做什么？

待办工具会修改 SQLite 数据库，因此它属于有副作用的工具。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
async def manage_todo(args: TodoArgs) -> list[dict[str, Any]] | dict[str, Any]:
    init_todo_db()
    with sqlite3.connect(TODO_DB) as conn:
        conn.row_factory = sqlite3.Row
        if args.action == 'add':
            if not args.title:
                raise ValueError('add 操作必须提供 title')
            cursor = conn.execute('INSERT INTO todos(title) VALUES (?)', (args.title,))
            return {'id': cursor.lastrowid, 'title': args.title, 'done': False}
        if args.action in {'complete', 'delete'} and not args.todo_id:
            raise ValueError(f'{args.action} 操作必须提供 todo_id')
        if args.action == 'complete':
            cursor = conn.execute('UPDATE todos SET done=1 WHERE id=?', (args.todo_id,))
            return {'updated': cursor.rowcount}
        if args.action == 'delete':
            cursor = conn.execute('DELETE FROM todos WHERE id=?', (args.todo_id,))
            return {'deleted': cursor.rowcount}
        rows = conn.execute('SELECT id, title, done FROM todos ORDER BY id').fetchall()
        return [dict(row) for row in rows]


### 现在做什么？

定义 `write_log`：先看输入，再看返回值，函数内部细节可以第二遍再读。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
async def write_log(args: LogArgs) -> dict[str, Any]:
    record = {
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'level': args.level,
        'message': args.message,
        'metadata': args.metadata,
    }
    path = WORKSPACE / 'agent.jsonl'
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')
    return {'written': True, 'path': str(path)}


### 现在做什么？

定义 `web_search`：先看输入，再看返回值，函数内部细节可以第二遍再读。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
async def web_search(args: SearchArgs) -> list[dict[str, str]]:
    async with httpx.AsyncClient(timeout=10, headers={'User-Agent': 'agent-learning-notebook/1.0'}) as http:
        response = await http.get('https://zh.wikipedia.org/w/api.php', params={
            'action': 'query', 'list': 'search', 'srsearch': args.query,
            'format': 'json', 'utf8': 1, 'srlimit': args.max_results,
        })
        response.raise_for_status()
        return [
            {'title': item['title'], 'snippet': item['snippet'], 'url': f"https://zh.wikipedia.org/wiki/{item['title'].replace(' ', '_')}"}
            for item in response.json()['query']['search']
        ]


### 现在做什么？

最后把上面的零件连接起来并做一次检查。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
registry = ToolRegistry()
for tool in [
    RegisteredTool('get_weather', '查询城市当前天气', WeatherArgs, get_weather),
    RegisteredTool('convert_currency', '按最新公开汇率换算货币', ExchangeArgs, convert_currency),
    RegisteredTool('manage_todo', '添加、列出、完成或删除待办', TodoArgs, manage_todo, side_effect=True),
    RegisteredTool('write_log', '写入一条结构化日志', LogArgs, write_log, side_effect=True),
    RegisteredTool('web_search', '搜索百科资料，返回标题、摘要和链接', SearchArgs, web_search),
]:
    registry.register(tool)

agent = MinimalAgent(registry)
print([schema['function']['name'] for schema in registry.schemas])


### 核心实验


### 现在做什么？

所有角色通过同一个结构化状态交接，避免只靠自然语言猜测对方输出。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
class WritingState(TypedDict, total=False):
    topic: str
    sources: list[dict[str, str]]
    draft: str
    review: dict[str, Any]
    revision_count: int
    status: str


### 现在做什么？

定义 `ask_bailian`：先看输入，再看返回值，函数内部细节可以第二遍再读。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
async def ask_bailian(system: str, user: str, temperature: float = 0.2) -> str:
    require_api_key()
    response = await client.chat.completions.create(
        model=BAILIAN_MODEL,
        messages=[{'role': 'system', 'content': system}, {'role': 'user', 'content': user}],
        temperature=temperature,
    )
    return response.choices[0].message.content or ''


### 现在做什么？

定义 `researcher_node`：先看输入，再看返回值，函数内部细节可以第二遍再读。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
async def researcher_node(state: WritingState) -> dict[str, Any]:
    results = await web_search(SearchArgs(query=state['topic'], max_results=5))
    return {'sources': results, 'status': 'writing'}


### 现在做什么？

定义 `writer_node`：先看输入，再看返回值，函数内部细节可以第二遍再读。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
async def writer_node(state: WritingState) -> dict[str, Any]:
    source_text = json.dumps(state['sources'], ensure_ascii=False)
    feedback = json.dumps(state.get('review', {}), ensure_ascii=False)
    draft = await ask_bailian(
        '你是严谨的中文作者。只能使用提供的来源陈述事实；用 [1] 形式引用，不得虚构来源。',
        f'主题：{state["topic"]}\n来源：{source_text}\n上一轮意见：{feedback}\n写一篇 600 字以内文章。',
    )
    return {'draft': draft, 'status': 'review'}


### 现在做什么？

定义 `reviewer_node`：先看输入，再看返回值，函数内部细节可以第二遍再读。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
async def reviewer_node(state: WritingState) -> dict[str, Any]:
    review_text = await ask_bailian(
        '你是审稿人。检查事实依据、引用、结构和表达。输出简短 JSON，字段 pass(boolean)、severity、issues(array)。',
        f'来源：{json.dumps(state["sources"], ensure_ascii=False)}\n初稿：{state["draft"]}',
    )
    try:
        cleaned = review_text.strip().removeprefix('```json').removesuffix('```').strip()
        review = json.loads(cleaned)
    except json.JSONDecodeError:
        review = {'pass': False, 'severity': 'medium', 'issues': ['审稿结果不是合法 JSON', review_text[:300]]}
    return {'review': review, 'revision_count': state.get('revision_count', 0) + 1}


### 现在做什么？

定义 `supervisor_route`：先看输入，再看返回值，函数内部细节可以第二遍再读。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
def supervisor_route(state: WritingState) -> Literal['done', 'rewrite', 'research']:
    if state['review'].get('pass') or state.get('revision_count', 0) >= 2:
        return 'done'
    if state['review'].get('severity') == 'high':
        return 'research'
    return 'rewrite'


### 现在做什么？

定义 `finish_writing`：先看输入，再看返回值，函数内部细节可以第二遍再读。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
def finish_writing(state: WritingState) -> dict[str, Any]:
    return {'status': 'done'}


### 现在做什么？

最后把上面的零件连接起来并做一次检查。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
writing_builder = StateGraph(WritingState)
writing_builder.add_node('researcher', researcher_node)
writing_builder.add_node('writer', writer_node)
writing_builder.add_node('reviewer', reviewer_node)
writing_builder.add_node('done', finish_writing)
writing_builder.add_edge(START, 'researcher')
writing_builder.add_edge('researcher', 'writer')
writing_builder.add_edge('writer', 'reviewer')
writing_builder.add_conditional_edges('reviewer', supervisor_route, {
    'done': 'done', 'rewrite': 'writer', 'research': 'researcher'
})
writing_builder.add_edge('done', END)
writing_team = writing_builder.compile(checkpointer=InMemorySaver())

# article = await writing_team.ainvoke(
#     {'topic': 'Agentic RAG 与普通 RAG 的差别', 'revision_count': 0},
#     {'configurable': {'thread_id': 'writing-demo-1'}},
# )
# print(article['draft'])
# print(article['review'])


## 3. 观察与验证

核心代码中的真实 API 调用默认被注释。先运行无需额度的断言或定义单元格；确认输出和预期一致后，再逐行取消示例注释。


## 4. 代码讲解

`revision_count` 是必要的循环保险。高严重度问题回到研究节点，普通表达问题只回到写作节点；审稿器返回非法 JSON 时也会得到可观察的失败结果。

调试建议：从报错的最后一行开始读，确认当前 Notebook 的单元格是否按顺序全部运行；若看到 `NameError`，通常是准备单元格未运行或内核已重启。


## 5. 常见问题

- **`ModuleNotFoundError`**：确认选中了 `.venv` 内核，并重新安装 `requirements.txt`。
- **提示未配置 API Key**：在启动 Jupyter 的同一个终端设置环境变量，然后重启内核。
- **网络超时或 429**：公开接口或模型服务可能限流；稍后重试，不要移除超时保护。
- **运行结果和预期不同**：先执行“Restart Kernel and Run All”，排除旧变量残留。
- **产生费用吗？**：只有实际调用百炼聊天或 Embedding 接口才会消耗额度；本地定义、SQLite 和断言不会。

## 6. 练习

- 用 Pydantic 严格校验 reviewer 输出
- 比较单 Agent 与团队方案的延迟
- 增加事实核验员但保持最大轮数不变

建议先复制相关单元格再修改，保留一份能工作的基线。


## 7. 本课验收

完成后逐项确认：

- [ ] 每个角色输入输出明确
- [ ] 返工最多两轮
- [ ] 来源不足时不会把问题伪装成通过

如果某项还解释不清，回到对应代码，用更小的输入单独调用函数，而不是直接运行完整 Agent。


## 下一步

继续学习 `09_评估与可观测性.ipynb`。

> 学习记录建议：写下今天最重要的一个概念、遇到的一个错误、以及你如何验证修复。


## 一句话回顾

多 Agent 的重点是分工和交接，不是创建很多聊天机器人。

### 如果你仍然觉得难

先只完成以下最低目标：

- 能从上到下运行本课；
- 能指出哪一格是输入、哪一格产生输出；
- 能用一句话说出本课解决了什么问题。

做到这三点就可以进入下一课。第二遍学习时再研究类型注解、异常分支和工程细节。
